# Prompt Chaining

## Introduction

![alt text](prompt_chaning.excalidraw.png)

In [2]:
# import your LLM model
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.1:8b')
results = llm.invoke('Hi')
print(results)

content="It's nice to meet you. Is there something I can help you with or would you like to chat?" additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2025-09-27T06:24:40.7345767Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1783048400, 'load_duration': 32375900, 'prompt_eval_count': 11, 'prompt_eval_duration': 96229300, 'eval_count': 23, 'eval_duration': 1653925600, 'model_name': 'llama3.1:8b'} id='run--79cb5488-827d-4d2d-b178-b555f7fe8922-0' usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34}


In [31]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display,Markdown
from langchain_core.runnables.graph import MermaidDrawMethod
import nest_asyncio
import os
nest_asyncio.apply()
os.environ["PYPPETEER_CHROMIUM_REVISION"] = "1181217"
os.environ["PYPPETEER_BROWSER_EXECUTABLE_PATH"] = r"C:\Users\Hanif\AppData\Local\pyppeteer\pyppeteer\chrome-win\chrome.exe"

# Define graph state
class State(TypedDict):
    topic: str 
    story: str
    criteria_story : str
    improved_story:str
    final_story:str

# Nodes

def generate_story(state:State):
    msg = llm.invoke(f"Write a one sentence story with premise about {state['topic']}")
    return {"story":msg.content}

def check_criteria(state: State):
    story_text = state['improved_story'] if state.get('improved_story') else state['story']
    msg = llm.invoke(
        f"Does the story contain three characters or more? "
        f"Answer with Failed or Passed only, don't use another word. Story: {story_text}"
    )
    return {"criteria_story": msg.content}
def improve_story(state:State):
    msg = llm.invoke(f"Improve the story by three characters or more. Story: {state['story']}")
    return {"improved_story":msg.content}

def final_user_story(state:State):
    final_story = state.get('improved_story') or state.get('story', '')
    return {"final_story": final_story}

In [32]:
graph = StateGraph(State)
# Define nodes
graph.add_node("generate",generate_story)
graph.add_node("criteria",check_criteria)
graph.add_node("improve",improve_story)
graph.add_node("final",final_user_story)

# Define edges
graph.add_edge(START,"generate")
graph.add_edge("generate", "criteria")
graph.add_conditional_edges(
    "criteria",
    lambda state: "Passed" if "Passed" in state["criteria_story"] else "Failed",
    {"Passed": "final", "Failed": "improve"}
)


graph.add_edge("improve", "criteria")



graph.add_edge("final",END) 

# Compile the graph
compiled_graph = graph.compile()

# Visualize the graph

mermaid_code = compiled_graph.get_graph().draw_mermaid()
with open("graph.mmd", "w") as f:
    f.write(mermaid_code)

In [33]:
# Stream to monitoring graph execution
config = {'configurable': {'thread_id': '2'}}
state={"topic":"Corporate success"}
for chunk in compiled_graph.stream(state,config,stream_mode='updates'):
    print(chunk)

{'generate': {'story': "After years of climbing the corporate ladder, Alexandra finally landed the coveted CEO position at Smith & Co., but soon discovered that her new title came with a steep price: sacrificing her own identity and values in favor of the company's ruthless pursuit of profit."}}
{'criteria': {'criteria_story': 'Failed'}}
{'improve': {'improved_story': 'Here is an improved story with three additional characters:\n\nAfter years of climbing the corporate ladder, Alexandra finally landed the coveted CEO position at Smith & Co., but soon discovered that her new title came with a steep price: sacrificing her own identity and values in favor of the company\'s ruthless pursuit of profit.\n\nHer old colleagues, who had been instrumental in her rise to power, seemed oblivious to the moral compromises she was making. Rachel, her former mentor and now head of marketing, would often whisper encouraging words about Alexandra\'s "vision" for the company, but remained mum on the true 

In [35]:
state={"topic":"How to be successful in corporate world?"}
result = compiled_graph.invoke(state)
print(result)

{'topic': 'How to be successful in corporate world?', 'story': "As she rose through the ranks of her company, Sarah realized that it wasn't just about the late nights and early mornings, but about building genuine relationships, staying adaptable, and learning from every failure along the way.", 'criteria_story': 'Passed', 'improved_story': 'Here is an improved version of the story with four new characters:\n\nAs Sarah rose through the ranks of her company, she quickly discovered that success was not solely dependent on her own efforts, but also on the people around her. Her mentor, Rachel, a seasoned executive, had taken her under her wing and taught her the importance of building genuine relationships in the workplace.\n\n"Relationships are key to your growth and mine," Rachel would often say. "Invest in them, nurture them, and they will reward you tenfold."\n\nSarah took Rachel\'s words to heart and began to focus on developing strong bonds with her colleagues. She started by gettin

In [46]:
print(f"initial story : {result['story']}")
print(f"final story : {result['final_story']}")

initial story : As she rose through the ranks of her company, Sarah realized that it wasn't just about the late nights and early mornings, but about building genuine relationships, staying adaptable, and learning from every failure along the way.
final story : Here is an improved version of the story with four new characters:

As Sarah rose through the ranks of her company, she quickly discovered that success was not solely dependent on her own efforts, but also on the people around her. Her mentor, Rachel, a seasoned executive, had taken her under her wing and taught her the importance of building genuine relationships in the workplace.

"Relationships are key to your growth and mine," Rachel would often say. "Invest in them, nurture them, and they will reward you tenfold."

Sarah took Rachel's words to heart and began to focus on developing strong bonds with her colleagues. She started by getting to know each of their stories, listening actively to their concerns, and offering suppor

## Parallelization

In [47]:
# import your LLM model
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.1:8b')
results = llm.invoke('Hello my name is Hanif')
print(results)

content='Nice to meet you, Hanif! Is there something I can help you with or would you like to chat for a bit?' additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2025-09-27T09:14:07.6013404Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8271508100, 'load_duration': 5965827000, 'prompt_eval_count': 16, 'prompt_eval_duration': 589070600, 'eval_count': 27, 'eval_duration': 1712003500, 'model_name': 'llama3.1:8b'} id='run--6b964882-8ac2-4065-ac5d-ea9a769c0bb4-0' usage_metadata={'input_tokens': 16, 'output_tokens': 27, 'total_tokens': 43}


In [48]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display,Markdown

class State(TypedDict):
    topic : str
    characters : str
    settings: str
    premises: str
    story_intro : str

In [49]:
def generate_characters(state:State):
    """ Generate characters based on description """
    msg = llm.invoke(f"Create two characters names and brief traits for story about {state['topic']}")
    return {"characters": msg.content}

def generate_settings(state:State):
    """Generate story settings"""
    msg = llm.invoke(f"Describe a vivid setting for story about {state['topic']}")
    return {"settings": msg.content}

def generate_premises(state:State):
    """Generate story premises"""
    msg = llm.invoke(f"Generate a one sentence plot premise for story about {state['topic']}")
    return {"premises": msg.content}

def combine_elements(state:State):
    """Combine characters, settings, and premises into a story intro"""
    msg = llm.invoke(
        f"Write a short story introduction using the following elements. "
        f"Using the characters: {state['characters']}, "
        f"settings: {state['settings']}, and "
        f"premises: {state['premises']}, "
        f"write a compelling story introduction."
    )
    return {"story_intro": msg.content}


In [50]:
graph = StateGraph(State)
graph.add_node("characters", generate_characters)
graph.add_node("settings", generate_settings)
graph.add_node("premises", generate_premises)
graph.add_node("combine", combine_elements)

graph.add_edge(START, "characters")
graph.add_edge(START, "settings")
graph.add_edge(START, "premises")
graph.add_edge("characters", "combine")
graph.add_edge("settings", "combine")
graph.add_edge("premises", "combine")
graph.add_edge("combine", END)

compiled_graph = graph.compile()
mermaid_code = compiled_graph.get_graph().draw_mermaid()
with open("prompt_chaining_graph.mmd", "w") as f:
    f.write(mermaid_code)

In [51]:
# Stream to monitoring graph execution
config = {'configurable': {'thread_id': '2'}}
state={"topic":"Good governance goverment"}
for chunk in compiled_graph.stream(state,config,stream_mode='updates'):
    print(chunk)

{'characters': {'characters': 'Here are two character profiles for a story about good governance in government:\n\n**Character 1:**\n\n* **Name:** Maya Jensen\n* **Age:** 35\n* **Occupation:** Deputy Minister of Public Administration\n* **Traits:** Maya is a no-nonsense, results-driven individual who believes that transparency and accountability are essential components of effective governance. She is well-organized, highly analytical, and has a keen eye for detail.\n* **Personality:** Maya is a natural leader, known for her confidence and charisma. She is fiercely independent and values fairness and justice above all else.\n\n**Character 2:**\n\n* **Name:** Ethan Patel\n* **Age:** 42\n* **Occupation:** Mayor of a large metropolitan city\n* **Traits:** Ethan is a charismatic, people-person who has spent his career building relationships with citizens, community leaders, and stakeholders. He is passionate about public service and genuinely cares about the well-being of those he serves.\